## 1 : Initialisation

In [ ]:
from huggingface_hub import snapshot_download
import transformers
from einops import rearrange


In [ ]:
snapshot_download("baffo32/decapoda-research-llama-7B-hf")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]


KeyboardInterrupt



In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "/root/.cache/huggingface/hub/models--baffo32--decapoda-research-llama-7B-hf/snapshots/aa18b48a1330572a6dd5f5d5619ed19838ca285c"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(device)
# decapoda's config has broken special token IDs
model.config.bos_token_id = 1
model.config.eos_token_id = 2
model.config.pad_token_id = 0


tokenizer = AutoTokenizer.from_pretrained(
    MODEL, unk_token="<unk>", bos_token="<s>", eos_token="</s>"
)
tokenizer.add_bos_token = True
if hasattr(tokenizer, "update_post_processor"):
    tokenizer.update_post_processor()

print(tokenizer("")["input_ids"])  # should print [1]


# LLaMA marks spaces with '▁', not GPT-2's 'Ġ'
model.to_str_tokens = lambda t: [t.replace('▁', ' ') for t in tokenizer.tokenize(t)]

model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[1]


## 2: Main

Q1 : run completion on the following sentence: "it is raining cats and dogs"

In [ ]:
type(tokenizer)

transformers.models.llama.tokenization_llama.LlamaTokenizer

In [ ]:
type(model)

transformers.models.llama.modeling_llama.LlamaForCausalLM

In [ ]:
prompt = "the cat sat on the mat"

inputs = tokenizer(prompt, return_tensors="pt")

token_ids = model.generate(inputs.input_ids, max_length=30)

input_ids = inputs['input_ids'].shape
print(input_ids)

# Generates sequences of token ids for models with a language modeling head.
token_ids = model.generate(inputs.input_ids, max_length=30)

tokenizer.batch_decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

torch.Size([1, 7])


'the cat sat on the mat.\nthe cat sat on the mat.\nthe cat sat on the mat.\nthe cat sat on the'

In [ ]:
attrs = [a for a in dir(model) if not a.startswith('_')]
attrs


['T_destination',
 'active_adapters',
 'add_adapter',
 'add_model_tags',
 'add_module',
 'adjust_generation_fn',
 'all_tied_weights_keys',
 'apply',
 'barrier_after_gathered_checkpoint_save',
 'base_model',
 'base_model_prefix',
 'bfloat16',
 'buffers',
 'call_super_init',
 'can_generate',
 'can_record_outputs',
 'children',
 'compile',
 'compute_transition_scores',
 'config',
 'config_class',
 'continuous_batching_context_manager',
 'cpu',
 'cuda',
 'delete_adapter',
 'dequantize',
 'destroy_cached_continuous_batching_manager',
 'device',
 'disable_adapters',
 'disable_input_require_grads',
 'double',
 'dtype',
 'dummy_inputs',
 'dump_patches',
 'enable_adapters',
 'enable_input_require_grads',
 'enable_peft_hotswap',
 'ep_plan',
 'eval',
 'extra_repr',
 'float',
 'forward',
 'from_pretrained',
 'fsdp_plan',
 'fsdp_size',
 'gather_sharded_state_dict_for_save',
 'generate',
 'generate_batch',
 'generation_config',
 'generation_config_class',
 'get_adapter_state_dict',
 'get_buffer',
 '

In [ ]:
help(model.generate)
help(tokenizer.batch_decode)

Help on method generate in module transformers.generation.utils:

generate(
    inputs: torch.Tensor | None = None,
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None,
    stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None,
    prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None,
    synced_gpus: bool | None = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: torch.Tensor | None = None,
    negative_prompt_attention_mask: torch.Tensor | None = None,
    custom_generate: str | collections.abc.Callable | None = None,
    **kwargs
) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generation.utils.GenerateEncoderDecoderOutput | 

## 3 Implement z steer



### Algorithm 1: Extracting $\mathbf{z}_{steer}$ for a sentence

**Input:**
- $x$ – target sentence
- $M$ – pretrained language model
- $\theta$ – pretrained language model weights
- $I_L$ – injection location
- $I_T$ – injection timestep
- $d$ – dimension of $\mathbf{z}_{steer}$

**Output:** $\mathbf{z}_{steer}$ – extracted candidate steering vector

1. $\mathbf{z}_{steer} \sim \text{xavier\_normal}(d)$
2. **for** $i \leftarrow [1, 2, \dots, N]$ **do**
3. &emsp; $\text{logits} = M_\theta.\text{forward}(x, \mathbf{z}_{steer}, I_L, I_T)$
4. &emsp; $\mathcal{L} = \text{XENT}(\text{logits}, x)$
5. &emsp; $\mathcal{L}.\text{backward}()$
6. &emsp; $\mathbf{z}_{steer} = \mathbf{z}_{steer} + lr \cdot \dfrac{\partial \mathcal{L}}{\partial \mathbf{z}_{steer}}$
7. **end**
8. **return** $\mathbf{z}_{steer}$


**Explain what entropy means when applied to a proability distribution**
- Entropy measures how uncertain you are about the outcome before it happens.
- Entropy for a die is highest when probaiblity is uniform. And the number of bits needed to represent the outcome is highest
- An event is more surprising the lower probabiltiy it is, and less surprsining the higher probability

**Explain what cross entropy is- **
- Cross entropy is a measure of the difference between two distributions


In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy

In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy

cfg = model.config
z_steer = t.empty(1, cfg.hidden_size, device=model.device)
xavier_normal_(z_steer)
z_steer.requires_grad_(True)

N = 3
model = model.to(device)
token_ids = token_ids.to(device)
lr = 0.01


for i in range(1, N):
  logits = model.forward(token_ids).logits
  logits = rearrange(logits, "b s v -> b v s")
  loss = cross_entropy(logits, token_ids)
  loss.backward()

  #update

  with torch.no_grad():
      z_steer += lr * z_steer.grad
  print(z_steer)


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 17.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.37 GiB is allocated by PyTorch, and 34.01 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
z_steer.grad

In [ ]:
logits

In [ ]:
token_ids.squeeze().shape

In [ ]:
type(token_ids)
type(logits)

In [ ]:
logits.shape

In [ ]:
token_ids.shape

In [ ]:
dir(transformers.modeling_outputs.CausalLMOutputWithPast)

In [ ]:
import numpy as np
# what is teh cross entropy between the two distrbituions?
x1 = t.tensor([0.1, 0.2, 0.3, 0.4, 0.5])
x2 = t.tensor([0.1, 0.2, 0.3, 0.4, 0.5])
cross_entropy(x1, x2)
# wht is teh cross entropy not 0 , the distribtions are indentical
x1 = t.tensor([0.1, 0.2, 0.8, 0.4, 0.5])
x2 = t.tensor([0.1,10 , 0.3, 0.4, 0.5])
cross_entropy(x1, x2)
c1 = t.tensor([0.4, 0.6])
c2 = t.tensor([0.7, 0.3])
cross_entropy(c1, c2)

In [ ]:
help(loss.backward)

In [ ]:
cfg


In [ ]:
help(model.forward)

In [ ]:
model.config

## 4 How to intervene on a model in pytorch

In [ ]:
## How to create a pytorch hook, and then modify say layer 7, and timestep 1 which corresponds to the token "cat"

In [ ]:
z_steer

Next task:

- Intervene on the model, add forward hook to attention layer 7. And add z_steer to it
- Measure outputs prior to intervention and post intervention

In [ ]:
cfg

In [ ]:
tokenizer.convert_ids_to_tokens(token_ids.squeeze())[6]

In [ ]:
help(tokenizer)

Experiment1: Add z_steer to l0 , what is the final predicted sequence

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

hook = partial(add_steering_vector, z_steer=z_steer, timestep = 6)

l0._forward_hooks.clear()
l0 = model.model.layers[0]
l0.register_forward_hook(hook)

model = model.to(device)
model(token_ids)

In [ ]:
z_steer.dtype

Aside 1 : How to perform inference using the model

In [ ]:
from torch.nn.functional import softmax
logits = model(token_ids).logits
probs = softmax(logits, dim = -1)
tokens = probs.argmax(dim=-1)
print(tokens.shape)
tokenizer.convert_ids_to_tokens(tokens.squeeze())

## 5 Perform gradient descent on z_steer

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out



In [ ]:
cfg = model.config
z_steer = t.empty(1, cfg.hidden_size, device=model.device)
xavier_normal_(z_steer)
z_steer.requires_grad_(True)

N = 500
model = model.to(device)
token_ids = token_ids.to(device)
lr = 0.01

# clear any outstanding hooks from previous runs
for m in model.modules():
     m._forward_hooks.clear()
     m._forward_pre_hooks.clear()

## add hook
hook = partial(add_steering_vector, z_steer=z_steer, timestep = 6)
l3 = model.model.layers[2]
handle = l3.register_forward_hook(hook)

cross_ent = []
for i in range(1, 500):
  logits = model(token_ids).logits
  logits = rearrange(logits, "b s v -> b v s")
  loss = cross_entropy(logits, token_ids)
  loss.backward()

  #update
  with torch.no_grad():
      z_steer -= lr * z_steer.grad

  #rest z_steer to 0
  z_steer.grad = None
  cross_ent.append(loss)

  if i % 20 == 0 :
     print(f"Completed 20 iterations, loss = {loss}")


handle.remove()

## 6 Load Datasets

Cornell movie dialogs corpus
https://convokit.cornell.edu/documentation/movie.html
- Stats: 1m load, 40mb
- docs: https://convokit.cornell.edu/documentation/movie.html

Nltk dataset
https://github.com/nltk/nltk_data
- Stats: 3s load, 4mb
- docs: https://www.nltk.org/

gigaword dataset
https://huggingface.co/datasets/walzen/gigaword
- stats: 13s load, 600mb

Wikitext 103
https://huggingface.co/datasets/Salesforce/wikitext
- 600mb


In [ ]:
##init

!pip install convokit nltk datasets


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 12.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 53.1 MB/s  0:00:00
  Created wheel for convokit: filename=convokit-5.0.0-py3-none-any.whl size=404788 sha256=f88c7396c36510a15037a5f3e0a46ff11abf4fd64eceec3d6475f14b84d40389
  Stored in directory: /root/.cache/pip/wheels/3c/a6/d0/b15d166a32d56d7c6f9463f6294f658c8eec7a4f7ee2920444
Successfully built convokit
  Attempting uninstall: h5py
    Found existing installation: h5py 3.14.0
    Uninstalling h5py-3.14.0:
      Successfully uninstalled h5py-3.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [convokit]


In [ ]:
## Loading all datasets

from convokit import Corpus, download
corpus = Corpus(filename=download("movie-corpus"))




No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem


In [ ]:
import nltk
nltk.download('gutenberg')

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Unzipping corpora/gutenberg.zip.


True

In [ ]:
from datasets import load_dataset

ds_gigaword = load_dataset("walzen/gigaword", split="validation")

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  296MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  296MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 29.4MB            

... (2 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3803957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/189651 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1951 [00:00<?, ? examples/s]

In [ ]:
ds_wikitext = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00001-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

... (2 outputs left out)

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

## 7 Dataset Processing

In [ ]:
import pandas as pd

## Dataset Construction Steps

1. **Load each corpus into a dataframe**, one row per document (or paragraph, or dialogue line), with columns `domain` and `text`.
2. **Concatenate** the four dataframes into one.
3. **Clean the `text` column**: remove headers, Gutenberg front matter, and formatting artifacts like ` @-@ `, then drop empty rows.
4. **Sentence-tokenize** each row's `text` with NLTK's `sent_tokenize`, giving a list of sentences per row.
5. **Explode** the sentence lists so each sentence becomes its own row, keeping its `domain`.
6. **Count words** in each sentence with NLTK's regex tokenizer, storing the count in a new column.
7. **Assign a length bin** (5–10, 10–15, …, 35–40, 40–128) based on the word count.
8. **Drop sentences outside 5–128 words.**
9. **Sample 8 sentences per (domain, bin)** with a fixed random seed, giving 64 per domain.
10. **Check and save**: confirm each (domain, bin) has 8 sentences, skim a few for leftover junk, and save the table to a file.

### (I) Process cornell movie dialogue

In [ ]:
df_cornell_movie = corpus.get_utterances_dataframe()

In [ ]:
df_cornell_movie.shape

(304713, 8)

In [ ]:
df_cornell_movie.columns

Index(['timestamp', 'text', 'speaker', 'reply_to', 'conversation_id',
       'meta.movie_id', 'meta.parsed', 'vectors'],
      dtype='object')

In [ ]:
df_cornell_movie.iloc[0]

,L1045
timestamp,None
text,They do not!
speaker,u0
reply_to,L1044
conversation_id,L1044
meta.movie_id,m0
meta.parsed,"[{'rt': 1, 'toks': [{'tok': 'They', 'tag': 'PR..."
vectors,[]


In [ ]:
df_cornell_movie.iloc[0:5]

,timestamp,text,speaker,reply_to,conversation_id,meta.movie_id,meta.parsed,vectors
id,,,,,,,,
L1045,None,They do not!,u0,L1044,L1044,m0,"[{'rt': 1, 'toks': [{'tok': 'They', 'tag': 'PR...",[]
L1044,None,They do to!,u2,None,L1044,m0,"[{'rt': 1, 'toks': [{'tok': 'They', 'tag': 'PR...",[]
L985,None,I hope so.,u0,L984,L984,m0,"[{'rt': 1, 'toks': [{'tok': 'I', 'tag': 'PRP',...",[]
L984,None,She okay?,u2,None,L984,m0,"[{'rt': 1, 'toks': [{'tok': 'She', 'tag': 'PRP...",[]
L925,None,Let's go.,u0,L924,L924,m0,"[{'rt': 0, 'toks': [{'tok': 'Let', 'tag': 'VB'...",[]


### (II)Process NLTK dataset

In [ ]:
## convert file into pandas dataframe

from nltk.corpus import gutenberg

help(gutenberg)


Help on PlaintextCorpusReader in module nltk.corpus.reader.plaintext object:

class PlaintextCorpusReader(nltk.corpus.reader.api.CorpusReader)
 |  PlaintextCorpusReader(
 |      root,
 |      fileids,
 |      word_tokenizer=WordPunctTokenizer(pattern='\\w+|[^\\w\\s]+', gaps=False, discard_empty=True, flags=re.UNICODE|re.MULTILINE|re.DOTALL),
 |      sent_tokenizer=None,
 |      para_block_reader=<function read_blankline_block at 0x7bd38f12e660>,
 |      encoding='utf8'
 |  )
 |
 |  Reader for corpora that consist of plaintext documents.  Paragraphs
 |  are assumed to be split using blank lines.  Sentences and words can
 |  be tokenized using the default tokenizers, or by custom tokenizers
 |  specified as parameters to the constructor.
 |
 |  This corpus reader can be customized (e.g., to skip preface
 |  sections of specific document formats) by creating a subclass and
 |  overriding the ``CorpusView`` class variable.
 |
 |  Method resolution order:
 |      PlaintextCorpusReader
 |   

In [ ]:
type(gutenberg)

In [ ]:
gutenberg

In [ ]:
import nltk
import pandas as pd
from nltk.corpus import gutenberg

nltk.download('gutenberg')
nltk.download('punkt_tab')

df_sents = pd.DataFrame(
    [(f, " ".join(s)) for f in gutenberg.fileids() for s in gutenberg.sents(f)],
    columns=["book", "sentence"]
)

print(df_sents.head())

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


              book                                           sentence
0  austen-emma.txt                       [ Emma by Jane Austen 1816 ]
1  austen-emma.txt                                           VOLUME I
2  austen-emma.txt                                          CHAPTER I
3  austen-emma.txt  Emma Woodhouse , handsome , clever , and rich ...
4  austen-emma.txt  She was the youngest of the two daughters of a...


### (III) Process gigaword

In [ ]:
from datasets.arrow_dataset import Dataset

In [ ]:
df_gigaword = ds_gigaword.to_pandas('/root/.cache/huggingface/datasets/walzen___gigaword/default/0.0.0/e038b4f0ef7315fef1d4eab772a487ec2cdb3a2c/gigaword-validation.arrow)')

In [ ]:
print(ds_gigaword.cache_files)

[{'filename': '/root/.cache/huggingface/datasets/walzen___gigaword/default/0.0.0/e038b4f0ef7315fef1d4eab772a487ec2cdb3a2c/gigaword-validation.arrow'}]


In [ ]:
ds_gigaword[0]

{'article': 'wes santee , a record-setting track star who never ran the UNK mile expected of him and whose running career was ended by a suspension for accepting too much expense money , died sunday at his home in eureka , kan. .',
 'summary': 'wes UNK record-setting runner dies at ##'}

### (IV) Processs wikitext

In [ ]:
print(ds_wikitext.cache_files)

[{'filename': '/root/.cache/huggingface/datasets/Salesforce___wikitext/wikitext-103-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext-validation.arrow'}]


In [ ]:
df_wikitext = ds_wikitext.to_pandas('/root/.cache/huggingface/datasets/Salesforce___wikitext/wikitext-103-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext-validation.arrow')

##8 Gigaword Dataset Construction

In [ ]:
from datasets import load_dataset
import nltk
nltk.download('punkt_tab')
ds_gigaword = load_dataset("walzen/gigaword", split="validation")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  296MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  296MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

... (3 outputs left out)

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3803957 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/189651 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1951 [00:00<?, ? examples/s]

In [ ]:
df_gigaword = ds_gigaword.to_pandas('/root/.cache/huggingface/datasets/walzen___gigaword/default/0.0.0/e038b4f0ef7315fef1d4eab772a487ec2cdb3a2c/gigaword-validation.arrow)')

In [ ]:
df_gigaword.shape, df_gigaword.columns

((189651, 2), Index(['article', 'summary'], dtype='object'))

In [ ]:
df_gigaword

,article,summary
0,"wes santee , a record-setting track star who n...",wes UNK record-setting runner dies at ##
1,in the signature scene of `` georgia '' _ a ne...,she acts she sings too
2,a mortar shell apparently intended for u.s. tr...,mortar rips market nine killed as u.n. reduces...
3,in the bitter budget war that now engulfs this...,nice guy panetta gets tough on budget
4,texas instruments inc. reported monday its sec...,texas instruments #q profit falls ## percent
...,...,...
189646,"almost ##,### gay athletes from around the wor...",thousands converge on sydney for the weeklong ...
189647,south africa had an enviable record in the aca...,s. africa leads in science says minister
189648,valencia announced that its veteran defenders ...,valencia renews contracts of veterans UNK djukic
189649,"for once , england coach sven-goran eriksson h...",eriksson has fully fit squad but still has sel...


In [ ]:
from pprint import pprint

pprint(df_gigaword['article'][0])
print("\n")
pprint(df_gigaword['summary'][0])


('wes santee , a record-setting track star who never ran the UNK mile expected '
 'of him and whose running career was ended by a suspension for accepting too '
 'much expense money , died sunday at his home in eureka , kan. .')


'wes UNK record-setting runner dies at ##'


Q: How to clean a dataframe

In [ ]:
## dropping empty cols
print(len(df_gigaword))
df_gigaword = df_gigaword.dropna()
print(len(df_gigaword))

189651
189651


### G1: Inspect dataframe

In [ ]:
def inspect_df(df, n):
  df = df.sample(n)
  articles = df['article']
  summaries = df['summary']

  for article, summary in zip(articles,summaries):
    print("article:")
    pprint(article)
    print('\n')
    print("summary:")
    pprint(summary)
    print('\n')
    print('------------------------------------')

inspect_df(df_gigaword, n=5)

article:
('kuomintang presidential candidate lien chan promised on tuesday to hold a '
 'conference on national reforms , in response to calls from academic circles '
 'for such a gathering .')


summary:
'lien promises to hold conference on national reforms'


------------------------------------
article:
("philips , europe 's largest maker of consumer electronics , said wednesday "
 'it will dissolve its components division , taking a ### million euro -lrb- '
 'us$ ### million -rrb- charge in the fourth quarter .')


summary:
'philips to dissolve components unit take ### million euro charge'


------------------------------------
article:
('more than ## years after a jogger was found raped , beaten and near death in '
 'a muddy central park ravine , prosecutors are trying to determine whether '
 'five teenagers were unjustly convicted for the crime .')


summary:
("prosecutor to seek more time to investigate man 's admission in #### central "
 'park jogger attack')


----------------

### G2: Tokenise text in gigaword, tokenize article col using nltk tokenizer

In [ ]:
from nltk import NLTKWordTokenizer,word_tokenize, sent_tokenize

s1 = "the cat sat on the mat"
tokenizer = NLTKWordTokenizer()

sent_tokenize(s1)



['the cat sat on the mat']

In [ ]:
help(sent_tokenize)

Help on function sent_tokenize in module nltk.tokenize:

sent_tokenize(text, language='english')
    Return a sentence-tokenized copy of *text*,
    using NLTK's recommended sentence tokenizer
    (currently :class:`.PunktSentenceTokenizer`
    for the specified language).

    :param text: text to split into sentences
    :param language: the model name in the Punkt corpus



In [ ]:
articles = df_gigaword["article"]

tokenized_articles = articles.apply(sent_tokenize)

In [ ]:
def inspect_series(s, n=5, seed=None):
    for idx, text in s.sample(n, random_state=seed).items():
        print(f"[{idx}]")
        pprint(text)
        print("-" * 40)

inspect_series(tokenized_articles, n=5)

[139258]
['trading volume in march rose from a year ago at the chicago board of trade '
 'and the chicago mercantile exchange amid uncertainty about the direction of '
 'u.s. interest rates .']
----------------------------------------
[130728]
['south african all-rounder lance klusener is out of the fourth test against '
 'england and will fly home for treatment to an injured ankle .']
----------------------------------------
[76384]
['brazilian defender cafu is ready to leave as roma for european champions ac '
 'milan , local daily la gazzetta dello sport reported on wednesday .']
----------------------------------------
[165770]
['president ma ying-jeou stressed the importance of innovation in creating '
 "what he calls a `` golden decade '' for taiwan in a speech delivered at a "
 'news conference wednesday on the eve of the second anniversary of his '
 'inauguration .']
----------------------------------------
[122590]
['nigerian author chinua achebe was awarded with the german bo

### G3: Count words in each sentence

In [ ]:
def count_words(s: list) -> int:
  """
  pass in a sentence and returns the number of words in that sentences
  """
  s = s[0]
  s = s.split()
  num = len(s)
  return num

sentence = ['chinese share prices closed sharply higher monday , adding #.## percent on '
 'news the government has selected another ## companies , including some of '
 "the country 's biggest blue chip names , to take part in the second phase of "
 'a pilot program for the sale of state-owned shares , dealers said .']

count_words(sentence)

53

In [ ]:
import pandas as pd

def create_df_with_counts(sentences: list[str]) -> pd.DataFrame:
    """
    Build a dataframe of sentences and their word counts.

    args:
        sentences: list of sentence strings
    returns:
        DataFrame with columns 'sentence' and 'count'
    """
    rows = [{"sentence": s[0], "count": count_words(s)} for s in sentences]
    return pd.DataFrame(rows)

df_with_counts = create_df_with_counts(tokenized_articles)

In [ ]:
df_with_counts

,sentence,count
0,"wes santee , a record-setting track star who n...",42
1,in the signature scene of `` georgia '' _ a ne...,67
2,a mortar shell apparently intended for u.s. tr...,46
3,in the bitter budget war that now engulfs this...,34
4,texas instruments inc. reported monday its sec...,36
...,...,...
189646,"almost ##,### gay athletes from around the wor...",35
189647,south africa had an enviable record in the aca...,34
189648,valencia announced that its veteran defenders ...,20
189649,"for once , england coach sven-goran eriksson h...",26


### G4 assing them into length bins

In [ ]:
def bin_by_word_count(df : pd.DataFrame) -> pd.DataFrame:
  """
  Take a dataframe with text, counts and add a bin column


  """
  counts = df['count']
  labels = ['5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40', '40-128']
  bins   = [5, 10, 15, 20, 25, 30, 35, 40, 129]

  series_bins = pd.cut(counts, bins = bins, labels = labels)
  df['bins'] = series_bins

  return df

df = bin_by_word_count(df_with_counts)

In [ ]:
df['count'].max()

95

In [ ]:
df

,sentence,count,bins
0,"wes santee , a record-setting track star who n...",42,40-128
1,in the signature scene of `` georgia '' _ a ne...,67,40-128
2,a mortar shell apparently intended for u.s. tr...,46,40-128
3,in the bitter budget war that now engulfs this...,34,30-35
4,texas instruments inc. reported monday its sec...,36,35-40
...,...,...,...
189646,"almost ##,### gay athletes from around the wor...",35,30-35
189647,south africa had an enviable record in the aca...,34,30-35
189648,valencia announced that its veteran defenders ...,20,15-20
189649,"for once , england coach sven-goran eriksson h...",26,25-30


### G5 Sample by bin

In [ ]:
def sample_sentences(df: pd.DataFrame, n: int = 8, seed: int = 0) -> pd.DataFrame:
    """
    Sample n sentences per length bin. With 8 bins and n=8, returns 64 sentences.
    """
    labels = ['5-10', '10-15', '15-20', '20-25', '25-30', '30-35', '35-40', '40-128']
    samples = []

    for label in labels:
        subset = df[df["bins"] == label]
        samples.append(subset.sample(n=n, random_state=seed))

    return pd.concat(samples, ignore_index=True)

sampled = sample_sentences(df)
sampled

,sentence,count,bins
0,`` we are the champions ! '',7,5-10
1,so what is alex UNK 's latest book about ?,10,5-10
2,have a question about the news ?,7,5-10
3,a federal judge friday rejected the rev.,7,5-10
4,"on april # , #### , capt.",7,5-10
...,...,...,...
59,"tony UNK , president and chief executive of ha...",45,40-128
60,"the asian development bank -lrb- adb -rrb- , a...",43,40-128
61,chinese ministry of health saturday reported n...,41,40-128
62,china 's top legislator wu bangguo said here s...,42,40-128


### G6 Sace to csv

In [ ]:
sampled.to_csv('64_sample_sentences.csv')


In [ ]:
x = pd.read_csv('64_sample_sentences.csv', index_col=0)

In [ ]:
x

,sentence,count,bins
0,`` we are the champions ! '',7,5-10
1,so what is alex UNK 's latest book about ?,10,5-10
2,have a question about the news ?,7,5-10
3,a federal judge friday rejected the rev.,7,5-10
4,"on april # , #### , capt.",7,5-10
...,...,...,...
59,"tony UNK , president and chief executive of ha...",45,40-128
60,"the asian development bank -lrb- adb -rrb- , a...",43,40-128
61,chinese ministry of health saturday reported n...,41,40-128
62,china 's top legislator wu bangguo said here s...,42,40-128


## 9 Compute BLEU-4 on the target sentence

Setup insturcitons:
Run sections 8,9

### G0 Run model on the sample sentences

In [ ]:
from huggingface_hub import snapshot_download
import transformers
from einops import rearrange

In [ ]:
snapshot_download("baffo32/decapoda-research-llama-7B-hf")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

'/root/.cache/huggingface/hub/models--baffo32--decapoda-research-llama-7B-hf/snapshots/aa18b48a1330572a6dd5f5d5619ed19838ca285c'

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "/root/.cache/huggingface/hub/models--baffo32--decapoda-research-llama-7B-hf/snapshots/aa18b48a1330572a6dd5f5d5619ed19838ca285c"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).to(device)
# decapoda's config has broken special token IDs
model.config.bos_token_id = 1
model.config.eos_token_id = 2
model.config.pad_token_id = 0


tokenizer = AutoTokenizer.from_pretrained(
    MODEL, unk_token="<unk>", bos_token="<s>", eos_token="</s>"
)
tokenizer.add_bos_token = True
if hasattr(tokenizer, "update_post_processor"):
    tokenizer.update_post_processor()

print(tokenizer("")["input_ids"])  # should print [1]


# LLaMA marks spaces with '▁', not GPT-2's 'Ġ'
model.to_str_tokens = lambda t: [t.replace('▁', ' ') for t in tokenizer.tokenize(t)]

model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got -1. This may result in unexpected behavior.
[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[1]


In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=31999)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-06)
      )
    )
    (norm): LlamaRMSNorm((409

In [ ]:
print(x['sentence'][23])
print(x['sentence'][24])
print(x['sentence'][25])
print(x['sentence'][26])
print(x['sentence'][27])

the nashville predators signed center greg johnson and defenseman bill houlder to contract extensions wednesday .
cup winners cup : red star belgrade # -lrb- dejan stankovic #th , ##th , zoran UNK ##th -rrb- , hjk helsinki # .
heavy firing was heard near sri lanka 's only international airport early monday following several explosions , officials and residents said .
the new york mercantile exchange said it will raise the margins on natural gas futures contracts at the close of trading monday .
zambia has said it can host the common market for eastern and southern africa -lrb- comesa -rrb- show next year .


In [ ]:
dir(transformers.models.llama.modeling_llama.LlamaForCausalLM)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_adjust_bias',
 '_adjust_missing_and_unexpected_keys',
 '_apply',
 '_assisted_decoding',
 '_auto_class',
 '_backward_compatibility_gradient_checkpointing',
 '_beam_search',
 '_beam_search_has_unfinished_sequences',
 '_call_impl',
 '_can_compile_fullgraph',
 '_can_record_outputs',
 '_can_set_attn_implementation',
 '_can_set_experts_implementation',
 '_can_set_experts_implementation_cached_value',
 '_check_and_adjust_attn_implementation',
 '_check_and_adjust_experts_implementati

In [ ]:
help(transformers.models.llama.modeling_llama.LlamaForCausalLM.generate)

Help on function generate in module transformers.generation.utils:

generate(
    self: 'GenerativePreTrainedModel',
    inputs: torch.Tensor | None = None,
    generation_config: transformers.generation.configuration_utils.GenerationConfig | None = None,
    logits_processor: transformers.generation.logits_process.LogitsProcessorList | None = None,
    stopping_criteria: transformers.generation.stopping_criteria.StoppingCriteriaList | None = None,
    prefix_allowed_tokens_fn: collections.abc.Callable[[int, torch.Tensor], list[int]] | None = None,
    synced_gpus: bool | None = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: torch.Tensor | None = None,
    negative_prompt_attention_mask: torch.Tensor | None = None,
    custom_generate: str | collections.abc.Callable | None = None,
    **kwargs
) -> transformers.generation.utils.GenerateDecoderOnlyOutput | transformers.generat

In [ ]:
help(AutoModelForCausalLM)

Help on class AutoModelForCausalLM in module transformers.models.auto.modeling_auto:

class AutoModelForCausalLM(transformers.models.auto.auto_factory._BaseAutoModelClass)
 |  AutoModelForCausalLM(*args, **kwargs) -> None
 |
 |  This is a generic model class that will be instantiated as one of the model classes of the library (with a causal language modeling head) when created
 |  with the [`~AutoModelForCausalLM.from_pretrained`] class method or the [`~AutoModelForCausalLM.from_config`] class
 |  method.
 |
 |  This class cannot be instantiated directly using `__init__()` (throws an error).
 |
 |  Method resolution order:
 |      AutoModelForCausalLM
 |      transformers.models.auto.auto_factory._BaseAutoModelClass
 |      builtins.object
 |
 |  Class methods defined here:
 |
 |  from_config(**kwargs) from transformers.models.auto.auto_factory._BaseAutoModelClass
 |      Instantiates one of the model classes of the library (with a causal language modeling head) from a configuration.
 

In [ ]:
help(transformers.models.llama.tokenization_llama.LlamaTokenizer)

Help on class LlamaTokenizer in module transformers.models.llama.tokenization_llama:

class LlamaTokenizer(transformers.tokenization_utils_tokenizers.TokenizersBackend)
 |  LlamaTokenizer(
 |      vocab: str | dict | list | None = None,
 |      merges: str | list | None = None,
 |      clean_up_tokenization_spaces=False,
 |      unk_token='<unk>',
 |      bos_token='<s>',
 |      eos_token='</s>',
 |      use_default_system_prompt=False,
 |      legacy=False,
 |      add_prefix_space=None,
 |      **kwargs
 |  )
 |
 |  Construct a Llama tokenizer. Based on byte-level Byte-Pair-Encoding.
 |
 |  This uses notably ByteFallback and no normalization.
 |
 |  ```python
 |  >>> from transformers import LlamaTokenizer
 |
 |  >>> tokenizer = LlamaTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")
 |  >>> tokenizer.encode("Hello this is a test")
 |  [1, 15043, 445, 338, 263, 1243]
 |  ```
 |
 |  If you want to change the `bos_token` or the `eos_token`, make sure to specify them when

In [ ]:
prompts = list(x['sentence'])
# promtps = list[str] , len = 64

In [ ]:
#64 sentences
import torch as t

def generate_completion(prompt: str):
  """
  Generates a sequence from one token_id
  e.g from the token : 4124 -> 4124 3219 0214 ... | the -> the cat sat
  arg:

  returns:
  Tensor() of token ids e.g [1, 3201, 4012 ...]
  """

  inputs = tokenizer(prompt, return_tensors="pt")
  start_token = inputs.input_ids[:,1:2] # shape(1,n) -> shape (1,1) , get second token from sequence
  seq_length = inputs.input_ids.shape[1]
  predicted_token_ids = model.generate(start_token, max_length= seq_length)
  return predicted_token_ids

def decode_completion(token_ids) -> str:
  """
  tokenizes a list of prompts into a list of token ids

  args:
  token_ids : tensor of ids

  returns:
  the predicted output in string format
  """
  completion = tokenizer.batch_decode(token_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
  return completion


prompt = prompts[1]
token_ids = generate_completion(prompt)
completion = decode_completion(token_ids) # pass in token at psn 1,1 in tensor, shape of token _ids = (1, n)
# Generates sequences of token ids for models with a language modeling head.

print(prompt)
print(completion) # exp we will win the game | along those lines



so what is alex UNK 's latest book about ?
so that the user can easily find the information they are looking for.


In [ ]:
completions[0]

NameError: name 'completions' is not defined

### G1 Try computing Bleu



Q: What is BLEU a measure of?

BLEU scores how closely generated text matches a reference by measuring the overlap of their 1- to 4-word sequences, with a penalty for outputs that are too short.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

# Define your desired weights (example: higher weight for bi-grams)
weights = (0.25, 0.25, 0, 0)  # Weights for uni-gram, bi-gram, tri-gram, and 4-gram

# Reference and predicted texts (same as before)
reference = [["the", "picture", "is", "clicked", "by", "me"],
             ["this", "picture", "was", "clicked", "by", "me"]]
predictions = ["the", "picture", "the", "picture", "by", "me"]

# Calculate BLEU score with weights
score = sentence_bleu(reference, predictions, weights=weights)
print(score)



In [ ]:
### G1 implement method1 Bleu-smoothing from https://aclanthology.org/W14-3346.pdf
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.tokenize import word_tokenize

def compute_bleu(reference: str, hypothesis: str) -> float:
    """
    Compute smoothed BLEU-4 between a reference and a hypothesis sentence.

    returns:
        bleu score (0 to 1)
    """
    smooth_func = SmoothingFunction()

    ref_tokens = word_tokenize(reference)                     # CHANGED: tokenize
    hyp_tokens = word_tokenize(hypothesis)                    # CHANGED: tokenize

    bleu4 = sentence_bleu([ref_tokens], hyp_tokens,           # CHANGED: reference in a list
                          smoothing_function=smooth_func.method1)

    print("reference")
    print(reference)
    print("hypothesis")
    print(hypothesis)
    print(f"Score: {bleu4}")
    print("--------------")
    print("\n")

    return bleu4                                              # ADDED: return the score



step 0, loss = 5.8789
step 20, loss = 1.8564
step 40, loss = 0.3176
step 60, loss = 0.0682
step 80, loss = 0.0322
step 100, loss = 0.0211
step 120, loss = 0.0153
step 140, loss = 0.0118
ref: `` we are the champions ! ''
hyp: `` we are the champions ! ''
reference
`` we are the champions ! ''
hypothesis
`` we are the champions ! ''
Score: 1.0
--------------


step 0, loss = 5.1875
step 20, loss = 1.4951
step 40, loss = 0.3625
step 60, loss = 0.1799
step 80, loss = 0.1127
step 100, loss = 0.0761
step 120, loss = 0.0516
step 140, loss = 0.0334
step 160, loss = 0.0224
step 180, loss = 0.0152
step 200, loss = 0.0105
ref: so what is alex UNK 's latest book about ?
hyp: so what is alex UNK 's latest book about ?
reference
so what is alex UNK 's latest book about ?
hypothesis
so what is alex UNK 's latest book about ?
Score: 1.0
--------------


step 0, loss = 4.5156
step 20, loss = 1.1533
step 40, loss = 0.0217
ref: have a question about the news ?
hyp: have a question about the news ?
refere

KeyboardInterrupt: 

### I1 The Average bleu score is very low, base generation doesnt contain any information about the target sentence

## Next steps
1. Use the steering vectors to help reconstruct target sentence
2. Calculate Bleu after using steering vectors

## 10 Injecting steering vector at different layers


### G1 Steer model on reference prompt

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy

def train_loop(model, token_ids, z_steer, lr, N):
  cross_ent = []
  for i in range(1, N):
    logits = model(token_ids).logits
    logits = rearrange(logits, "b s v -> b v s")
    loss = cross_entropy(logits, token_ids)
    loss.backward()

    #update
    with torch.no_grad():
        z_steer -= lr * z_steer.grad

    #rest z_steer to 0
    z_steer.grad = None
    cross_ent.append(loss)

    if i % 20 == 0 :
      print(f"Completed 20 iterations, loss = {loss}")
  return cross_ent

def steer_model(model, reference, int_layer= 6):
  """
  Fine tune the model on the task of reconstructing a sentence.
  Model is intervened on at layer n. We add z_steer which is updated with gradient descent
  over each intervention, onto layer n.

  returns:
  model with z_steer intervention
  """
  cfg = model.config
  z_steer = t.empty(1, cfg.hidden_size, device=model.device)
  xavier_normal_(z_steer)
  z_steer.requires_grad_(True)

  prompt = " ".join(reference)
  inputs = tokenizer(prompt, return_tensors="pt")
  token_ids = inputs.input_ids

  ## hyperparams
  N = 500
  lr = 0.01
  epochs = 1

  #move tensor and model to gpu
  model = model.to(device)
  token_ids = token_ids.to(device)


  # clear any outstanding hooks from previous runs
  for m in model.modules():
      m._forward_hooks.clear()
      m._forward_pre_hooks.clear()

  # add hook
  hook = partial(add_steering_vector, z_steer=z_steer, timestep = 6)
  layer = model.model.layers[int_layer]
  handle = layer.register_forward_hook(hook)

  for epoch in range(0,epochs):
    cross_ent = train_loop(model,token_ids, z_steer, lr, N)

  return cross_ent

prompt = prompts[1]
print(prompt)
cross_ent = steer_model(model, prompt)

  # handle.remove()

so what is alex UNK 's latest book about ?
Completed 20 iterations, loss = 6.82421875
Completed 20 iterations, loss = 6.82421875
Completed 20 iterations, loss = 6.82421875
Completed 20 iterations, loss = 6.82421875
Completed 20 iterations, loss = 6.82421875
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8203125
Completed 20 iterations, loss = 6.8125
Completed 20 iterations, loss = 6.8125
Completed 20 iterations, loss = 6.8125
Completed 20 iterations, loss = 6.8125
Completed 20 iterations, loss = 6.8125
Completed 20 iterations, loss = 6.80859375
Completed 20 iterations, loss = 6.80859375
Completed 20 iterations, loss = 6.80859375
Completed 20 iterations, loss = 6.80859375
Completed 20 iterations, loss = 6.80859

1. Next step, use ADAM instead of SGD
2. Calculate Bleu score, using the updated model

### G2 Implement ADAM optimiser

In [ ]:
## how2 access the module list -> how 2 access a single LlamaDecoderLayer
from functools import partial

# res stream : batch, psn , dmodel  | 1, 30, 4096

def add_steering_vector(inst, inp, out, z_steer, timestep):
  z_steer = z_steer.to(out.dtype)
  # print(f"z steer : {z_steer.shape} , {z_steer.mean()}")
  # print(f"out before steering : {out.shape}, {out.mean()}")
  # out = batch, psn , model
  out[:, timestep] =  out[:, timestep] + z_steer
  # print(f"out after sterring : {out.shape}, {out.mean()}")
  return out

In [ ]:
help(torch.optim.Optimizer)

Help on class Optimizer in module torch.optim.optimizer:

class Optimizer(builtins.object)
 |  Optimizer(
 |      params: collections.abc.Iterable[torch.Tensor] | collections.abc.Iterable[dict[str, typing.Any]] | collections.abc.Iterable[tuple[str, torch.Tensor]],
 |      defaults: dict[str, typing.Any]
 |  ) -> None
 |
 |  Base class for all optimizers.
 |
 |  .. warning::
 |      Parameters need to be specified as collections that have a deterministic
 |      ordering that is consistent between runs. Examples of objects that don't
 |      satisfy those properties are sets and iterators over values of dictionaries.
 |
 |  Args:
 |      params (iterable): an iterable of :class:`torch.Tensor` s or
 |          :class:`dict` s. Specifies what Tensors should be optimized.
 |      defaults: (dict): a dict containing default values of optimization
 |          options (used when a parameter group doesn't specify them).
 |
 |  Methods defined here:
 |
 |  __getstate__(self) -> dict[str, typing

In [ ]:
import torch as t
from torch.nn.init import xavier_normal_
from torch.nn.functional import cross_entropy
from torch.optim import Adam                                   # ADAM: import

def train_loop(model, token_ids, z_steer, N, optimizer):       # ADAM: takes optimizer instead of lr
    cross_ent = []
    for i in range(N):
        logits = model(token_ids).logits
        pred = rearrange(logits[:, :-1], "b s v -> b v s")     # FIX: shift, predictions 0..n-2
        target = token_ids[:, 1:]                              # FIX: shift, the next tokens
        loss = cross_entropy(pred, target)
        loss.backward()

        optimizer.step()                                       # ADAM: replaces the manual update
        optimizer.zero_grad()                                  # ADAM: resets z_steer.grad

        cross_ent.append(loss.item())                          # FIX: store number, not tensor
        if i % 20 == 0:
            print(f"step {i}, loss = {loss.item():.4f}")
        if loss.item() < 0.01:                                 # early stop once learned
            break
    return cross_ent

def steer_model(model, reference, int_layer=6):
    """
    Optimize a steering vector so the frozen model reconstructs `reference`.
    z_steer is added to the output of layer `int_layer` at the first timestep.

    returns:
        cross_ent: list of losses
        z_steer: the trained steering vector
    """
    model = model.to(device)
    model.requires_grad_(False)                                # FIX: freeze the model

    cfg = model.config
    z_steer = t.empty(1, cfg.hidden_size, device=device)
    xavier_normal_(z_steer)
    z_steer.requires_grad_(True)

    inputs = tokenizer(reference, return_tensors="pt")         # FIX: no " ".join on a string
    eos = t.tensor([[tokenizer.eos_token_id]])
    token_ids = t.cat([inputs.input_ids, eos], dim=1).to(device)

    N = 500
    lr = 0.1
    optimizer = Adam([z_steer], lr=lr)                         # ADAM: create optimizer

    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    hook = partial(add_steering_vector, z_steer=z_steer, timestep=0)   # FIX: 6 → 0
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        cross_ent = train_loop(model, token_ids, z_steer, N, optimizer)
    finally:
        handle.remove()                                        # FIX: always remove the hook

    return cross_ent, z_steer.detach()

prompt = prompts[1]
print(prompt)
cross_ent, z_steer = steer_model(model, prompt)
print(cross_ent[-5:])                                          # should end near 0

so what is alex UNK 's latest book about ?
step 0, loss = 5.1836
step 20, loss = 1.4834
step 40, loss = 0.3755
step 60, loss = 0.1875
step 80, loss = 0.1175
step 100, loss = 0.0805
step 120, loss = 0.0555
step 140, loss = 0.0366
step 160, loss = 0.0244
step 180, loss = 0.0169
step 200, loss = 0.0117
[0.01068115234375, 0.01047515869140625, 0.01029205322265625, 0.01013946533203125, 0.00994873046875]


In [ ]:
z_steer

tensor([[-2.2706, -1.1142, -2.9228,  ..., -0.0974, -1.9084,  0.3907]],
       device='cuda:0')

### G3 calculate blue using updated model

In [ ]:
def generate_completion(max_new_tokens: int = 100):
    """
    Greedily generate a sequence starting from the <s> token.
    Any steering hook attached to the model is applied during generation.

    returns:
        Tensor of token ids, e.g. [[1, 3201, 4012, ..., 2]]
    """
    start_token = t.tensor([[tokenizer.bos_token_id]], device=device)

    return model.generate(
        start_token,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=False,                    # keeps the hook's position 0 = <s> on every step
    )

In [ ]:
@t.no_grad()
def perform_inference(z_steer, reference, int_layer=6, timestep=0):
    for m in model.modules():
        m._forward_hooks.clear()
        m._forward_pre_hooks.clear()

    z_steer = z_steer.to(device)                              # move first
    hook = partial(add_steering_vector, z_steer=z_steer, timestep=timestep)
    handle = model.model.layers[int_layer].register_forward_hook(hook)

    try:
        token_ids = generate_completion()           # make sure it uses use_cache=False
    finally:
        handle.remove()
    hypothesis = decode_completion(token_ids)
    print("ref:", reference)
    print("hyp:", hypothesis)
    return compute_bleu(reference, hypothesis)

print(perform_inference(z_steer, prompts[1]))

ref: so what is alex UNK 's latest book about ?
hyp: so what is alex UNK 's latest book about ?
reference
so what is alex UNK 's latest book about ?
hypothesis
so what is alex UNK 's latest book about ?
Score: 0.009170625612054907
--------------


None


In [ ]:
### computes bleu
def compute_all_bleu(prompts, n=8, int_layer=6, timestep=0):
    scores = []
    vectors = {}

    for prompt in prompts[:n]:
        cross_ent, z_steer = steer_model(model, prompt, int_layer=int_layer)
        score = perform_inference(z_steer, prompt, int_layer=int_layer, timestep=timestep)

        scores.append(score)
        vectors[prompt] = z_steer.cpu()          # keep each sentence's vector

    print(f"Mean BLEU over {len(scores)} sentences: {sum(scores) / len(scores):.4f}")
    return scores, vectors

scores, vectors = compute_all_bleu(prompts)

### Errors so far and tips
# Steering Vector Replication: Lessons Learned

## Notebook hygiene

- **Stale globals:** a variable not defined inside a function silently falls back to an old value from an earlier cell (this caused both the `token_ids` and `hypothesis` bugs). Run **Runtime → Restart session and run all** regularly to surface these as errors.
- **Rerun edited cells:** changing a function's code does nothing until you rerun the cell that defines it, and then any cells that use it.
- **Leftover hooks:** every `register_forward_hook` call stacks another hook. Clear hooks first, and remove them in a `try`/`finally`.

## Model loading and memory

- **Freeze the model** with `model.requires_grad_(False)`; otherwise gradients for all weights cause out-of-memory errors.
- **Load in half precision**, and prefer **bfloat16** over fp16 for optimization, since fp16 can make gradients vanish and the loss look frozen.
- **Old checkpoints can have broken configs**, like decapoda LLaMA's empty special tokens. Set them explicitly or use a correctly configured copy.
- **Each model has its own special tokens:** LLaMA uses `<s>` (1) and `</s>` (2), while GPT-2 uses one `<|endoftext|>` for both.

## Hooks

- **Call `model(x)`, not `model.forward(x)`**, or hooks are skipped.
- **The hook's first argument is the module**, then inputs, then output. The hidden states are in `out` (or `out[0]` if it's a tuple).
- **Cast to the hidden states' dtype** (`z_steer.to(h.dtype)`), keep `z_steer` itself in fp32, and **`.clone()`** before editing the output in place.
- **Use `partial(fn, z_steer=z_steer)` with keyword names**; positional arguments land in the wrong parameters.
- **Index positions as `h[:, pos]`**, not `h[pos]`, which selects a batch item.
- **KV caching breaks fixed-position injection** during generation, so use `use_cache=False` or a manual greedy loop.

## Training the steering vector

- **Shift the loss:** compare `logits[:, :-1]` with `token_ids[:, 1:]`. Without it, the vector learns the wrong task.
- **`cross_entropy` needs the vocabulary in dimension 1**, so flatten or rearrange to `(batch, vocab, seq)`.
- **Append `</s>` to the target** so the vector learns where to stop.
- **Pass optimizers a list:** `Adam([z_steer], lr=lr)`.
- **Store `loss.item()`, not `loss`**, to avoid keeping every computation graph in memory.
- **Match training and inference settings exactly:** same layer, same timestep, same hook.
- **If the loss falls smoothly but slowly, raise the learning rate**; if it's frozen, check gradient norms.

## Generation and evaluation

- **Generate from the start token only** for reconstruction; passing the full sentence just continues it.
- **`max_length` includes the prompt**, so use `max_new_tokens`.
- **Batching sentences of different lengths needs padding** (left-padded for generation), or process them one at a time.
- **Decoding isn't generating:** `decode_completion` only turns IDs into text; generation must happen first.
- **BLEU needs token lists, not strings:** `sentence_bleu([ref_tokens], hyp_tokens)`. Strings silently give character-level nonsense.
- **Sanity-check metrics:** a sentence scored against itself must give 1.0.
- **Functions should `return` results**, not just print them.

## Python and pandas gotchas

- **`" ".join(s)` on a string spaces out its characters**; it's only for lists of words.
- **Tensor `.to(device)` isn't in place**: write `x = x.to(device)`.
- **Integer indexing drops a dimension** (`[:, 2]`), while slicing keeps it (`[:, 2:3]`).
- **Build dataframes from a list of rows**, not by assigning inside a loop.
- **Save CSVs with `index=False`** to avoid `Unnamed: 0` columns.

## Replication practice

- **Write down every choice the paper leaves open**: tokenizer, bin edges, BLEU smoothing, dataset cleaning, learning rate.
- **Debug on the paper's setup first** (GPT-2 small) when possible, then scale up.

G4 compare bleu on steered